In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [2]:
from src.transforms import test_transforms
from src.dataset import ImageDataset
import torch

annot_path = Path("../data/preprocessed/trainval/annotations.csv")
img_dir = Path("../data/preprocessed/trainval/images")

# test dataset without transforms
dataset = ImageDataset(annot_path, img_dir)
transformed_dataset = ImageDataset(annot_path, img_dir, test_transforms)

In [41]:
_, truth_labels = next(iter(transformed_dataset))

In [77]:
from src.utilities import convert_xywh_coordinates

coords = convert_xywh_coordinates(label_1)
coords

(tensor(10136.), tensor(5600.), tensor(42504.), tensor(50848.))

In [42]:
truth_labels.shape

torch.Size([5, 5])

In [5]:
IoU(label_1, label_1), GIoU(label_1, label_1)

(tensor(1.), tensor(1.))

In [6]:
bbox_1 = torch.tensor([2, 2, 3, 3])
bbox_2 = torch.tensor([4, 1, 5, 3])

IoU(bbox_1, bbox_2), GIoU(bbox_1, bbox_2)

(tensor(0.), tensor(-0.5000))

In [44]:
class_preds = torch.randn(100, 21)

In [47]:
import torch.nn.functional as F

class_costs = [-F.softmax(class_preds, dim=-1).select(-1, idx).unsqueeze(-1)
                   for idx in range(truth_labels.shape[0])]

In [49]:
len(class_costs)

5

In [52]:
torch.cat(class_costs, dim=-1).shape

torch.Size([100, 5])

In [65]:
bbox_preds = torch.randn(100, 4)
bbox_preds[0], truth_labels[0][1:]

(tensor([-0.8571,  0.3196,  0.2228,  1.6904]),
 tensor([0.5848, 0.7321, 0.1205, 0.3393]))

In [75]:
l1_costs = [(bbox_preds - truth_labels[i][1:]).abs().sum(-1, True) for i in range(truth_labels.shape[0])]
torch.cat(l1_costs, dim=-1).shape

torch.Size([100, 5])

In [76]:
l1_costs

[tensor([[3.3078],
         [6.4919],
         [1.7950],
         [2.2280],
         [3.8601],
         [4.0928],
         [1.4082],
         [3.6594],
         [4.2784],
         [4.6801],
         [3.5529],
         [5.3460],
         [4.0268],
         [1.8413],
         [2.0389],
         [5.2492],
         [3.6219],
         [4.5781],
         [4.7742],
         [1.8744],
         [1.8248],
         [3.2643],
         [1.3424],
         [4.0625],
         [2.8399],
         [2.6969],
         [3.7074],
         [2.9971],
         [2.4461],
         [3.7411],
         [3.1466],
         [2.1581],
         [6.4478],
         [4.3959],
         [1.4778],
         [4.8166],
         [4.4382],
         [3.3953],
         [3.2051],
         [1.9881],
         [2.1206],
         [3.6223],
         [1.9605],
         [1.7389],
         [3.2727],
         [5.1325],
         [2.7750],
         [3.0965],
         [5.1988],
         [2.2801],
         [4.0022],
         [2.6098],
         [4.

In [66]:
subtracted = (bbox_preds - truth_labels[0][1:])
subtracted.shape, subtracted

(torch.Size([100, 4]),
 tensor([[-1.4419e+00, -4.1252e-01,  1.0226e-01,  1.3511e+00],
         [-1.1812e+00, -1.8957e+00, -1.9537e+00,  1.4612e+00],
         [-3.2457e-01, -1.0377e+00, -1.1272e-01,  3.2002e-01],
         [-1.0462e+00, -9.1087e-02,  3.1611e-01,  7.7462e-01],
         [ 7.9539e-02, -1.2445e+00,  7.2996e-01,  1.8060e+00],
         [-2.4335e-01, -1.4817e+00, -9.6022e-01, -1.4075e+00],
         [-4.1444e-01, -1.8068e-01, -5.1310e-01,  2.9996e-01],
         [ 3.8489e-02, -1.8500e+00, -9.4332e-01,  8.2758e-01],
         [-1.8747e+00, -1.2046e+00,  8.6716e-01,  3.3194e-01],
         [-8.6707e-01, -5.2055e-01, -1.3341e+00, -1.9584e+00],
         [-4.3361e-01, -2.1403e+00, -5.1226e-01, -4.6664e-01],
         [-2.0759e+00, -1.7786e+00,  6.5180e-01,  8.3961e-01],
         [-1.1287e+00, -1.6339e+00, -5.7741e-01, -6.8678e-01],
         [ 5.1334e-01,  2.7416e-01,  1.6268e-01, -8.9113e-01],
         [-1.2437e+00, -3.3697e-01,  1.3441e-01, -3.2384e-01],
         [-3.1017e-01, -2.6962e+

In [67]:
absoluted = subtracted.abs()
absoluted

tensor([[1.4419e+00, 4.1252e-01, 1.0226e-01, 1.3511e+00],
        [1.1812e+00, 1.8957e+00, 1.9537e+00, 1.4612e+00],
        [3.2457e-01, 1.0377e+00, 1.1272e-01, 3.2002e-01],
        [1.0462e+00, 9.1087e-02, 3.1611e-01, 7.7462e-01],
        [7.9539e-02, 1.2445e+00, 7.2996e-01, 1.8060e+00],
        [2.4335e-01, 1.4817e+00, 9.6022e-01, 1.4075e+00],
        [4.1444e-01, 1.8068e-01, 5.1310e-01, 2.9996e-01],
        [3.8489e-02, 1.8500e+00, 9.4332e-01, 8.2758e-01],
        [1.8747e+00, 1.2046e+00, 8.6716e-01, 3.3194e-01],
        [8.6707e-01, 5.2055e-01, 1.3341e+00, 1.9584e+00],
        [4.3361e-01, 2.1403e+00, 5.1226e-01, 4.6664e-01],
        [2.0759e+00, 1.7786e+00, 6.5180e-01, 8.3961e-01],
        [1.1287e+00, 1.6339e+00, 5.7741e-01, 6.8678e-01],
        [5.1334e-01, 2.7416e-01, 1.6268e-01, 8.9113e-01],
        [1.2437e+00, 3.3697e-01, 1.3441e-01, 3.2384e-01],
        [3.1017e-01, 2.6962e+00, 1.3746e+00, 8.6817e-01],
        [5.8916e-01, 5.1693e-01, 6.4704e-01, 1.8688e+00],
        [1.111

In [68]:
summed = absoluted.sum(dim=-1)
summed

tensor([3.3078, 6.4919, 1.7950, 2.2280, 3.8601, 4.0928, 1.4082, 3.6594, 4.2784,
        4.6801, 3.5529, 5.3460, 4.0268, 1.8413, 2.0389, 5.2492, 3.6219, 4.5781,
        4.7742, 1.8744, 1.8248, 3.2643, 1.3424, 4.0625, 2.8399, 2.6969, 3.7074,
        2.9971, 2.4461, 3.7411, 3.1466, 2.1581, 6.4478, 4.3959, 1.4778, 4.8166,
        4.4382, 3.3953, 3.2051, 1.9881, 2.1206, 3.6223, 1.9605, 1.7389, 3.2727,
        5.1325, 2.7750, 3.0965, 5.1988, 2.2801, 4.0022, 2.6098, 4.5345, 1.4441,
        2.3991, 1.9219, 2.9160, 3.9457, 2.2230, 3.0387, 3.4424, 5.6961, 3.8935,
        4.2814, 5.2630, 2.5280, 4.1891, 3.1089, 4.6236, 3.3497, 2.5529, 4.9755,
        2.0148, 2.4587, 2.1651, 1.5381, 3.6030, 2.8209, 2.1196, 2.1704, 4.2581,
        4.6376, 4.1652, 3.0429, 2.2612, 3.2601, 2.4903, 5.9048, 3.3412, 4.4018,
        2.2303, 4.6863, 2.4255, 1.8688, 5.8099, 2.2589, 2.8550, 2.1577, 5.6794,
        5.6836])

In [70]:
summed.unsqueeze(-1).shape

torch.Size([100, 1])